In [10]:
import pandas as pd

# ─────────────────────────────────────────────
# 1. LOAD OOS PREDICTIONS
# ─────────────────────────────────────────────
ml = pd.read_csv("ml_test_predictions.csv", parse_dates=["date"])
dl = pd.read_csv("dl_test_predictions.csv", parse_dates=["date"])

# ─────────────────────────────────────────────
# 2. NORMALIZE FOLD COLUMN NAME
# ─────────────────────────────────────────────
def normalize_fold_col(df, source):
    if "fold_id" in df.columns:
        return df
    elif "fold" in df.columns:
        df = df.rename(columns={"fold": "fold_id"})
        return df
    else:
        raise ValueError(f"{source} file must contain 'fold_id' or 'fold'")

ml = normalize_fold_col(ml, "ML")
dl = normalize_fold_col(dl, "DL")

# ─────────────────────────────────────────────
# 3. BASIC SANITY CHECKS
# ─────────────────────────────────────────────
required_cols = {"date", "fold_id", "model", "y_true", "y_pred"}

assert required_cols.issubset(ml.columns), f"ML file missing columns: {required_cols - set(ml.columns)}"
assert required_cols.issubset(dl.columns), f"DL file missing columns: {required_cols - set(dl.columns)}"

# Ensure fold_id is integer (prevents merge bugs later)
ml["fold_id"] = ml["fold_id"].astype(int)
dl["fold_id"] = dl["fold_id"].astype(int)

# Optional but useful for diagnostics
ml["model_type"] = "ML"
dl["model_type"] = "DL"

# ─────────────────────────────────────────────
# 4. CONCATENATE
# ─────────────────────────────────────────────
test_all = (
    pd.concat([ml, dl], ignore_index=True)
      .sort_values(["date", "model"])
      .reset_index(drop=True)
)

# ─────────────────────────────────────────────
# 5. FINAL CONSISTENCY CHECKS
# ─────────────────────────────────────────────
# Each model should have at most one prediction per date
dup = test_all.groupby(["model", "date"]).size().max()
assert dup == 1, f"Duplicate predictions detected (max per model/date = {dup})"

# y_true should be identical across models for a given date
ytrue_std = test_all.groupby("date")["y_true"].std().max()
assert ytrue_std < 1e-6, f"Inconsistent y_true across models (max std = {ytrue_std})"

# ─────────────────────────────────────────────
# 6. SAVE
# ─────────────────────────────────────────────
test_all = test_all.drop(columns=["ticker"], errors="ignore")
test_all.to_csv("all_models_predictions.csv", index=False)

print("Saved: all_models_predictions.csv")
print(f"Rows: {len(test_all):,}")
print("Models:", sorted(test_all["model"].unique()))

Saved: all_models_predictions.csv
Rows: 5,816
Models: ['CNN_LSTM', 'LSTM', 'LightGBM', 'RandomForest', 'TCN', 'TFT', 'Transformer', 'XGBoost']
